# 01. Lecke - Bevezetés az AI ügynökökbe

Üdvözlünk az első leckében a **Kezdőknek szóló AI ügynökök** tanfolyamon!

Egy **AI ügynök** egy olyan program, amely egy nagy nyelvi modellt (LLM) használ érvelési motorjaként, és képes *műveleteket* végrehajtani a valós világban — API-k hívása, adatbázisok lekérdezése vagy kód futtatása — egy cél elérése érdekében a felhasználó nevében.

Ebben a jegyzetfüzetben elkészíted az első ügynöködet: egy **utazási ügynököt**, ami nyaralási célpontokat ajánl. Közben megtanulod, hogyan kell:

1. Csatlakozni a Microsoft Foundry Agent Service-hez a **Microsoft Agent Framework** segítségével.
2. Eszközt adni az ügynöknek — egy egyszerű Python függvényt, amit meghívhat.
3. Futtatni az ügynököt és megvizsgálni a válaszát.
4. Tokénról tokénre streamelni az ügynök válaszát.


## Beállítás

Mielőtt futtatnád ezt a jegyzetfüzetet, győződj meg róla, hogy:

1. **Van egy Microsoft Foundry projekted** egy telepített csevegőmodelllel (pl. `gpt-5-mini`).
2. **Be vagy jelentkezve az Azure CLI-vel** — futtasd az `az login` parancsot a terminálodban.
3. **Beállítottad a szükséges környezeti változókat:**
   - `AZURE_AI_PROJECT_ENDPOINT` — a Microsoft Foundry projekted végpontja.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — a telepített modell neve.

Az alábbi cella telepíti a szükséges Python csomagokat.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Első ügynököd létrehozása

Egy ügynöknek két dologra van szüksége:

- **Utasításokra**, amelyek megmondják neki, *ki ő* és *hogyan viselkedjen* (egy rendszer parancs).
- **Eszközökre** — Python függvényekre, amelyeket `@tool` díszítéssel láttunk el, és amelyeket az ügynök hívhat meg információk lekérésére vagy műveletek végrehajtására.

Lent definiálunk egy egyszerű eszközt, amely egy népszerű nyaralóhelyek listáját adja vissza. Az ügynök ezt az eszközt fogja használni, amikor a felhasználó utazási ajánlásokat kér.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Válaszok streamelése

Egy interaktívabb élmény érdekében az ügynök válaszát **streamelhetjük**. A teljes válaszra való várakozás helyett az ügynök a létrejövő szövegrészleteket adja át. Ez különösen hasznos csevegőfelületeken, ahol valós időben szeretnéd megjeleníteni a kimenetet.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Összefoglaló

Ebben a leckében megtanultad, hogyan:

- **Létrehozz egy szolgáltatót**, amely a Microsoft Foundry Agent Service-hez csatlakozik a `FoundryChatClient` segítségével.
- **Definiálsz egy eszközt** az `@tool` dekorátor használatával, hogy az ügynök hívhassa a Python függvényeidet.
- **Futtatod az ügynököt** egy felhasználói üzenettel, és kiíratod a válaszát.
- **Folyamatosan közvetíted a válaszokat** valós idejű kimenethez.

A következő leckében mélyebben megvizsgáljuk az ügynök keretrendszereket, és megtanuljuk, hogyan adhatsz az ügynököknek erőteljesebb eszközöket és többlépéses érvelési képességeket.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
